In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from sklearn.feature_selection import SelectFromModel
import joblib

# Set the folder containing datasets
folder_path = "Dataset/"  # Replace with your folder path
model_save_path = "models"  # Directory to save trained models
os.makedirs(model_save_path, exist_ok=True)  # Create directory if it doesn't exist

# Loop through each dataset in the folder
for file_name in os.listdir(folder_path):
    if file_name.endswith(".csv"):
        # Step 1: Load dataset
        dataset_path = os.path.join(folder_path, file_name)
        print(f"Processing dataset: {file_name}")
        df = pd.read_csv(dataset_path)

        # Step 2: Separate features (X) and target (y)
        X = df.iloc[:, :-1]  # All columns except the last one (features)
        y = df.iloc[:, -1]   # Last column (target)

        # Step 3: Encode non-numeric columns
        for col in X.select_dtypes(include=['object']).columns:
            le = LabelEncoder()
            X[col] = le.fit_transform(X[col])

        # Encode target label if it is categorical
        if y.dtype == 'object':
            y = LabelEncoder().fit_transform(y)

        # Step 4: Split data into training and testing sets
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        # Step 5: Handle imbalance in target using SMOTE
        print("Before SMOTE: Class distribution in y_train:", pd.Series(y_train).value_counts())
        smote = SMOTE(random_state=42)
        X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
        print("After SMOTE: Class distribution in y_train_resampled:", pd.Series(y_train_resampled).value_counts())

        # Step 6: Feature selection using Random Forest
        print("Performing feature selection using Random Forest...")
        rf_feature_selector = RandomForestClassifier(random_state=42)
        rf_feature_selector.fit(X_train_resampled, y_train_resampled)

        # Select features based on importance
        feature_selector = SelectFromModel(rf_feature_selector, threshold="mean", prefit=True)
        X_train_selected = feature_selector.transform(X_train_resampled)
        X_test_selected = feature_selector.transform(X_test)
        selected_features = X.columns[feature_selector.get_support()]
        print(f"Selected features: {list(selected_features)}")

        # Step 7: Train final Random Forest model with selected features
        print("Training Random Forest model with selected features...")
        rf_model = RandomForestClassifier(random_state=42)
        rf_model.fit(X_train_selected, y_train_resampled)

        # Step 8: Evaluate model
        y_pred = rf_model.predict(X_test_selected)
        y_proba = rf_model.predict_proba(X_test_selected)[:, 1]

        accuracy = accuracy_score(y_test, y_pred)
        roc_auc = roc_auc_score(y_test, y_proba)
        conf_matrix = confusion_matrix(y_test, y_pred)

        print(f"Accuracy: {accuracy}")
        print(f"ROC AUC: {roc_auc}")
        print("Confusion Matrix:")
        print(conf_matrix)
        print("Classification Report:")
        print(classification_report(y_test, y_pred))

        # Step 9: Save model
        model_file_name = f"{file_name.split('.')[0]}_RandomForest_RFFS.pkl"
        model_path = os.path.join(model_save_path, model_file_name)
        joblib.dump(rf_model, model_path)
        print(f"Model saved: {model_path}\n")


Processing dataset: ds_100K20.csv
Before SMOTE: Class distribution in y_train: 0    51017
1    29044
Name: phishing, dtype: int64
After SMOTE: Class distribution in y_train_resampled: 0    51017
1    51017
Name: phishing, dtype: int64
Performing feature selection using Random Forest...


C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(


Selected features: ['url_length', 'n_dots', 'n_hypens', 'n_slash']
Training Random Forest model with selected features...
Accuracy: 0.8733013589128698
ROC AUC: 0.9468141124500457
Confusion Matrix:
[[10961  1737]
 [  799  6519]]
Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.86      0.90     12698
           1       0.79      0.89      0.84      7318

    accuracy                           0.87     20016
   macro avg       0.86      0.88      0.87     20016
weighted avg       0.88      0.87      0.87     20016

Model saved: models\ds_100K20_RandomForest_RFFS.pkl

Processing dataset: ds_10K18.csv
Before SMOTE: Class distribution in y_train: 1    4012
0    3988
Name: Label, dtype: int64
After SMOTE: Class distribution in y_train_resampled: 1    4012
0    4012
Name: Label, dtype: int64
Performing feature selection using Random Forest...


C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(


Selected features: ['Domain', 'URL_Length', 'URL_Depth', 'Prefix/Suffix']
Training Random Forest model with selected features...
Accuracy: 0.959
ROC AUC: 0.9871491494775247
Confusion Matrix:
[[993  19]
 [ 63 925]]
Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.98      0.96      1012
           1       0.98      0.94      0.96       988

    accuracy                           0.96      2000
   macro avg       0.96      0.96      0.96      2000
weighted avg       0.96      0.96      0.96      2000

Model saved: models\ds_10K18_RandomForest_RFFS.pkl

Processing dataset: ds_10K50_rev.csv
Before SMOTE: Class distribution in y_train: 0    4012
1    3988
Name: CLASS_LABEL, dtype: int64


...

Selected features: ['NumDots', 'UrlLength', 'PathLevel', 'PathLength']
Training Random Forest model with selected features...
Accuracy: 0.7876304529916465
ROC AUC: 0.8827026447809182
Confusion Matrix:
[[87771 24809]
 [ 3334 16605]]
Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.78      0.86    112580
           1       0.40      0.83      0.54     19939

    accuracy                           0.79    132519
   macro avg       0.68      0.81      0.70    132519
weighted avg       0.88      0.79      0.81    132519

Model saved: models\ds_600K11_rev_RandomForest_RFFS.pkl

Processing dataset: ds_88K112.csv
Before SMOTE: Class distribution in y_train: 0    46388
1    24529
Name: phishing, dtype: int64
After SMOTE: Class distribution in y_train_resampled: 0    46388
1    46388
Name: phishing, dtype: int64
Performing feature selection using Random Forest...


C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(


Selected features: ['qty_slash_url', 'length_url', 'qty_dot_domain', 'qty_vowels_domain', 'domain_length', 'qty_dot_directory', 'qty_hyphen_directory', 'qty_underline_directory', 'qty_slash_directory', 'qty_equal_directory', 'qty_at_directory', 'qty_and_directory', 'qty_exclamation_directory', 'qty_tilde_directory', 'qty_comma_directory', 'qty_asterisk_directory', 'qty_dollar_directory', 'qty_percent_directory', 'directory_length', 'qty_dot_file', 'qty_slash_file', 'qty_questionmark_file', 'qty_at_file', 'qty_exclamation_file', 'qty_space_file', 'qty_tilde_file', 'qty_comma_file', 'qty_plus_file', 'qty_asterisk_file', 'file_length', 'time_response', 'asn_ip', 'time_domain_activation', 'time_domain_expiration', 'qty_nameservers', 'qty_mx_servers', 'ttl_hostname']
Training Random Forest model with selected features...
Accuracy: 0.9672306824591088
ROC AUC: 0.9943410126733659
Confusion Matrix:
[[11244   368]
 [  213  5905]]
Classification Report:
              precision    recall  f1-score

C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(


Selected features: ['Domain', 'DNSRecordType', 'StrangeCharacters', 'ConsoantRatio', 'NumericRatio', 'VowelRatio', 'NumericSequence', 'DomainLength']
Training Random Forest model with selected features...
Accuracy: 0.9999444444444444
ROC AUC: 0.9999444668673548
Confusion Matrix:
[[8854    0]
 [   1 9145]]
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      8854
           1       1.00      1.00      1.00      9146

    accuracy                           1.00     18000
   macro avg       1.00      1.00      1.00     18000
weighted avg       1.00      1.00      1.00     18000

Model saved: models\ds_90K32_RandomForest_RFFS.pkl



In [2]:
# Import necessary libraries
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from imblearn.over_sampling import SMOTE

# Directory containing datasets
input_folder = "dataset/"  # Replace with your folder path
output_csv = "modelRFFS_RFrfe.csv"

# Initialize a list to store results
results = []

# Loop through all files in the folder
for filename in os.listdir(input_folder):
    if filename.endswith(".csv"):  # Process only CSV files
        dataset_path = os.path.join(input_folder, filename)
        
        # Load dataset
        print(f"Processing {filename}...")
        df = pd.read_csv(dataset_path)

        # Separate features (X) and target (y)
        X = df.iloc[:, :-1]
        y = df.iloc[:, -1]
        
        # Encode non-numeric columns
        for col in X.select_dtypes(include=['object']).columns:
            X[col] = pd.factorize(X[col])[0]
        
        # Encode target if it is categorical
        if y.dtype == 'object':
            y = pd.factorize(y)[0]

        # Handle imbalance in target using SMOTE
        smote = SMOTE(random_state=42)
        X_resampled, y_resampled = smote.fit_resample(X, y)

        # Split into training and testing sets
        X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

        # Feature selection using RFE with Random Forest
        rf_for_rfe = RandomForestClassifier(random_state=42)
        rfe = RFE(estimator=rf_for_rfe, n_features_to_select=10)  # Adjust the number of features to select
        rfe.fit(X_train, y_train)

        # Get selected features
        selected_features = X.columns[rfe.support_]
        X_train_selected = rfe.transform(X_train)
        X_test_selected = rfe.transform(X_test)

        # Train final Random Forest model on selected features
        rf_model = RandomForestClassifier(random_state=42)
        rf_model.fit(X_train_selected, y_train)

        # Evaluate the model
        y_pred = rf_model.predict(X_test_selected)
        y_proba = rf_model.predict_proba(X_test_selected)[:, 1]
        
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
        roc_auc = roc_auc_score(y_test, y_proba)
        
        # Confusion matrix to calculate false positive rate
        conf_matrix = confusion_matrix(y_test, y_pred)
        false_positive_rate = conf_matrix[0, 1] / conf_matrix[0].sum() if conf_matrix[0].sum() > 0 else 0

        # Save results
        results.append({
            "Dataset Name": filename,
            "Number of Rows": df.shape[0],
            "Number of Columns": df.shape[1],
            "Number of Selected Features": len(selected_features),
            "Selected Features": ', '.join(selected_features),
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "False Positive Rate": false_positive_rate,
            "ROC AUC": roc_auc
        })

# Save results to a CSV file
results_df = pd.DataFrame(results)
results_df.to_csv(output_csv, index=False)

print(f"Results saved to {output_csv}")


Processing ds_100K20.csv...
Processing ds_10K18.csv...
Processing ds_10K50_rev.csv...
Processing ds_11055.csv...
Processing ds_11055_rev.csv...
Processing ds_11K89.csv...
Processing ds_129K112.csv...
Processing ds_235795_54_rev.csv...
Processing ds_247950_rev.csv...
Processing ds_600K11_rev.csv...


C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\sklearn\feature_selection\_rfe.py:291: UserWarning: Found n_features_to_select=10 > n_features=9. There will be no feature selection and all features will be kept.
  warnings.warn(


Processing ds_88K112.csv...
Processing ds_90K32.csv...
Results saved to modelRFFS_RFrfe.csv


In [3]:
# Import necessary libraries
import os
import time  # Import time for measuring runtime
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from imblearn.over_sampling import SMOTE

# Directory containing datasets
input_folder = "dataset/"  # Replace with your folder path
output_csv = "modelRFFS_RFrfe.csv"

# Initialize a list to store results
results = []

# Loop through all files in the folder
for filename in os.listdir(input_folder):
    if filename.endswith(".csv"):  # Process only CSV files
        dataset_path = os.path.join(input_folder, filename)
        
        # Start time for processing the dataset
        start_time = time.time()

        # Load dataset
        print(f"Processing {filename}...")
        df = pd.read_csv(dataset_path)

        # Separate features (X) and target (y)
        X = df.iloc[:, :-1]
        y = df.iloc[:, -1]
        
        # Encode non-numeric columns
        for col in X.select_dtypes(include=['object']).columns:
            X[col] = pd.factorize(X[col])[0]
        
        # Encode target if it is categorical
        if y.dtype == 'object':
            y = pd.factorize(y)[0]

        # Handle imbalance in target using SMOTE
        smote = SMOTE(random_state=42)
        X_resampled, y_resampled = smote.fit_resample(X, y)

        # Split into training and testing sets
        X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

        # Feature selection using RFE with Random Forest
        rf_for_rfe = RandomForestClassifier(random_state=42)
        rfe = RFE(estimator=rf_for_rfe, n_features_to_select=10)  # Adjust the number of features to select
        rfe.fit(X_train, y_train)

        # Get selected features
        selected_features = X.columns[rfe.support_]
        X_train_selected = rfe.transform(X_train)
        X_test_selected = rfe.transform(X_test)

        # Train final Random Forest model on selected features
        rf_model = RandomForestClassifier(random_state=42)
        rf_model.fit(X_train_selected, y_train)

        # Evaluate the model
        y_pred = rf_model.predict(X_test_selected)
        y_proba = rf_model.predict_proba(X_test_selected)[:, 1]
        
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
        roc_auc = roc_auc_score(y_test, y_proba)
        
        # Confusion matrix to calculate false positive rate
        conf_matrix = confusion_matrix(y_test, y_pred)
        false_positive_rate = conf_matrix[0, 1] / conf_matrix[0].sum() if conf_matrix[0].sum() > 0 else 0

        # End time for processing the dataset
        end_time = time.time()
        runtime = end_time - start_time

        # Save results
        results.append({
            "Dataset Name": filename,
            "Number of Rows": df.shape[0],
            "Number of Columns": df.shape[1],
            "Number of Selected Features": len(selected_features),
            "Selected Features": ', '.join(selected_features),
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "False Positive Rate": false_positive_rate,
            "ROC AUC": roc_auc,
            "Runtime (seconds)": runtime
        })

# Save results to a CSV file
results_df = pd.DataFrame(results)
results_df.to_csv(output_csv, index=False)

print(f"Results saved to {output_csv}")


Processing ds_100K20_.csv...
Processing ds_10K18.csv...
Processing ds_10K50_rev.csv...
Processing ds_11055.csv...
Processing ds_11055_rev.csv...
Processing ds_11K89.csv...
Processing ds_129K112.csv...
Processing ds_235795_54_rev.csv...
Processing ds_247950_rev.csv...
Processing ds_600K11_rev.csv...


C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\sklearn\feature_selection\_rfe.py:291: UserWarning: Found n_features_to_select=10 > n_features=9. There will be no feature selection and all features will be kept.
  warnings.warn(


Processing ds_88K112.csv...
Processing ds_90K32.csv...
Results saved to modelRFFS_RFrfe.csv
